An animation is a **sequence of frames**, where each frame corresponds to a plot on a Figure.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.animation as animation

##### **ANIMATION CLASSES**

`FuncAnimation`<br/>
Generate data for the first frame and then modify this data for each frame to create an animated plot.<br/>
Is more **efficient** in terms of speed and memory since it only draws an artist once, and then modifies it.

In [ ]:
%matplotlib widget
fig, ax = plt.subplots(figsize=(7,5))

t = np.linspace(0, 3, 40)
g = -9.81
v0 = 12
z = g * t**2 / 2 + v0 * t

v02 = 5
z2 = g * t**2 / 2 + v02 * t

# Save the created artists returned by the plot functions
# in order to modify them in the update() function.
scat = ax.scatter(t[0], z[0], c="b", s=5, label=f"v0 = {v0} m/s")
line2 = ax.plot(t[0], z2[0], label=f"v0 = {v02} m/s")[0]
ax.set(xlim=[0, 3], ylim=[-4, 10], xlabel="Time [s]", ylabel="Z [m]")
ax.legend()
ax.annotate(
    r"$z = g \frac{t^2}{2} + v t$",
    xy=(0,0),
    xytext=(0.5,0.5),
    xycoords="data",
    textcoords="axes fraction",
    size=20,

)


def update(frame):
    """
    This function, passed to the FuncAnimation, iteratively modifies the data of the plot.

    This is done by using setter methods of each artist e.g. set_xdata/set_data for a Line2D artist
    or set_offsets for a PathCollection (returned by the scatter method).
    """
    # for each frame, update the data stored on each artist
    x = t[:frame]
    y = z[:frame]

    # update the scatter plot
    data = np.stack([x, y]).T
    scat.set_offsets(data)

    # update the line plot
    line2.set_xdata(t[:frame])
    line2.set_ydata(z2[:frame])

    return (scat, line2)


# frames:   the length of the animation
# interval: time in ms between two successive frames
ani = animation.FuncAnimation(fig=fig, func=update, frames=40, interval=30)
plt.show()

`ArtistAnimation`<br/>
Generate a list (iterable) of artists that will draw in each frame in the animation.<br/>
Is more flexible since it allows any sequence of artists to be animated.

In [ ]:
%matplotlib widget
fig, ax = plt.subplots()

# Construct a new Generator
rng = np.random.default_rng(19680801)
# Data values (on the x-axis for barh)
data = np.array([20, 20, 20, 20])
# Bar positions (on the y-axis for barh)
x = np.array([1, 2, 3, 4])

artists = []
colors = ["blue", "red", "green", "purple"]

for i in range(20):
    # Add random values (from 0-9) to the data values
    data += rng.integers(low=0, high=10, size=data.shape)
    container = ax.barh(x, data, color=colors)
    # Store this frame as an artist object
    artists.append(container)

ani = animation.ArtistAnimation(fig=fig, artists=artists, interval=300)
plt.show()

##### **FASTER RENDERING BY USING BLITTING**

`Blitting` speeds up repetitive drawing by rendering all <span style="font-style:italic;">non-changing graphic elements</span> into a background image once.<br/>
Then, for every draw, only the changing elements need to be drawn onto this background.<br/>
The strategy is:
1. Prepare the constant background
    - Draw the figure excluding the artists you want to animate by marking them as **animated** (set_animated).
    - Save a copy of the RBGA buffer.
2. Render the individual images
    - Restore the RBGA buffer
    - Redraw the animated artists (draw_artists)
    - Show the resulting image on the screen

In [2]:
from matplotlib.backend_bases import FigureCanvasBase

FigureCanvasBase.supports_blit

False